# Create HRSA Awards from HRSA Data Warehouse

Creates Health Resources and Services Administration (HRSA) award rows from HRSA's own Data Warehouse awarded-grants CSV.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/hrsa_to_s3.py` to download the HRSA CSV, write parquet, and upload it.
- Contractor has prepared the script, notebook, and QA cells but does not have S3/Databricks access.
- The downloader writes raw/source columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does all type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** https://data.hrsa.gov/DataDownload/DD_Files/FS_EHB_AWARD_GRANT_FA_AGR_MVX.csv  
**S3 location:** `s3a://openalex-ingest/awards/hrsa/hrsa_projects.parquet`  
**Raw source shape validated locally on 2026-05-15:** 71,889 rows, 44 columns, award years 2016-2026, amount present on 100% of rows.

**HRSA funder:**
- funder_id: 4320332175
- ROR: https://ror.org/033jnv181
- DOI: 10.13039/100000102
- display_name: "Health Resources and Services Administration"

**Important source note:** `grant_number` repeats because the HRSA source contains annual financial-assistance rows. Do not deduplicate by `grant_number` alone. The downloader adds `source_row_hash`; this notebook uses `grant_number + award_year + source_row_hash` as the row-level native award key to avoid collapsing valid rows.


## Step 1: Create Staging Table from S3

In [ ]:
%sql
-- Create the staging table from HRSA Data Warehouse parquet.
CREATE OR REPLACE TABLE openalex.awards.hrsa_awards_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/hrsa/hrsa_projects.parquet`;


In [ ]:
%sql
-- Check raw row count. Local validation of the source CSV found 71,889 rows.
SELECT COUNT(*) as total_awards FROM openalex.awards.hrsa_awards_raw;


In [ ]:
%sql
-- Sample the raw HRSA data.
SELECT
    grant_number,
    award_year,
    financial_assistance,
    grant_program_name,
    hrsa_program_area_name,
    project_period_start_date,
    grant_project_period_end_date,
    grantee_name,
    abstract,
    source_row_hash
FROM openalex.awards.hrsa_awards_raw
LIMIT 10;


## Step 1.5: Inspect Raw Data, Money, Currency, and Native Keys

The HRSA CSV uses `financial_assistance` as the amount column. Currency is implicit USD because HRSA is a U.S. federal agency and the source does not publish a separate currency field. The raw source has repeated `grant_number` values, so key inspection is mandatory before transformation.


In [ ]:
%sql
-- Money-field scan per runbook §1.5.
SELECT column_name FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'hrsa_awards_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|obligated|outlay|grant|award|assistance|financial';


In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_amount,
    ROUND(SUM(CASE WHEN TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE) IS NOT NULL THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 1) AS pct_with_amount,
    MIN(TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE)) AS max_amount,
    SUM(TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE)) AS total_amount,
    SUM(CASE WHEN TRY_TO_DATE(project_period_start_date, 'MM/dd/yyyy') IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_start_date,
    SUM(CASE WHEN TRY_TO_DATE(grant_project_period_end_date, 'MM/dd/yyyy') IS NOT NULL THEN 1 ELSE 0 END) AS rows_with_end_date
FROM openalex.awards.hrsa_awards_raw;


In [ ]:
%sql
-- Native-key inspection: grant_number repeats, so the row-level key includes award_year and source_row_hash.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT grant_number) AS distinct_grant_numbers,
    COUNT(DISTINCT CONCAT_WS(':', LOWER(TRIM(grant_number)), TRIM(award_year))) AS distinct_grant_years,
    COUNT(DISTINCT CONCAT_WS(':', LOWER(TRIM(grant_number)), TRIM(award_year), source_row_hash)) AS distinct_row_level_keys
FROM openalex.awards.hrsa_awards_raw;


## Step 1.6: Fail-fast — Verify the HRSA Funder Row Exists

The transform does a `CROSS JOIN` to `openalex.common.funder` by `funder_id = 4320332175`. If that row is absent, the transform would silently emit zero rows. Stop if this query returns anything other than exactly one HRSA row.


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320332175;


## Step 2: Create HRSA Awards Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.hrsa_awards
USING delta
AS
WITH
hrsa_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320332175
),

raw_prepared AS (
    SELECT
        *,
        CONCAT_WS(':', LOWER(TRIM(grant_number)), TRIM(award_year), source_row_hash) AS row_level_award_id,
        TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE) AS parsed_amount,
        TRY_TO_DATE(project_period_start_date, 'MM/dd/yyyy') AS parsed_start_date,
        TRY_TO_DATE(grant_project_period_end_date, 'MM/dd/yyyy') AS parsed_end_date
    FROM openalex.awards.hrsa_awards_raw
    WHERE grant_number IS NOT NULL
      AND TRIM(grant_number) != ''
      AND award_year IS NOT NULL
      AND TRIM(award_year) != ''
      AND source_row_hash IS NOT NULL
      AND TRIM(source_row_hash) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.row_level_award_id))) % 9000000000 as id,

        COALESCE(NULLIF(TRIM(g.grant_program_name), ''), NULLIF(TRIM(g.hrsa_program_area_name), ''), g.grant_number) as display_name,

        CASE
            WHEN g.abstract IS NULL OR TRIM(g.abstract) = '' OR LOWER(TRIM(g.abstract)) = 'no link' THEN NULL
            ELSE g.abstract
        END as description,

        f.funder_id,
        g.row_level_award_id as funder_award_id,

        g.parsed_amount as amount,
        'USD' as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        CASE
            WHEN LOWER(COALESCE(g.grant_program_name, g.hrsa_program_area_name, '')) RLIKE 'training|traineeship|residency|workforce|scholarship|fellowship' THEN 'training'
            WHEN LOWER(COALESCE(g.grant_program_name, g.hrsa_program_area_name, '')) RLIKE 'cooperative' THEN 'grant'
            ELSE 'grant'
        END as funding_type,

        COALESCE(NULLIF(TRIM(g.grant_program_name), ''), NULLIF(TRIM(g.hrsa_program_area_name), '')) as funder_scheme,

        'hrsa_data_warehouse' as provenance,

        g.parsed_start_date as start_date,
        g.parsed_end_date as end_date,
        YEAR(g.parsed_start_date) as start_year,
        YEAR(g.parsed_end_date) as end_year,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        CAST(NULL AS STRING) as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.row_level_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN hrsa_funder f
)

SELECT * FROM awards_transformed;


In [ ]:
%sql
-- Remove previous HRSA Data Warehouse data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'hrsa_data_warehouse' AND priority = 57;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    57 as priority
FROM openalex.awards.hrsa_awards;


## Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_hrsa_awards FROM openalex.awards.hrsa_awards;


In [ ]:
%sql
-- Confirm the transformed table has no extra raw/helper columns.
DESCRIBE openalex.awards.hrsa_awards;


In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date
FROM openalex.awards.hrsa_awards
LIMIT 10;


In [ ]:
%sql
-- Check funder distribution. Should be only HRSA.
SELECT funder.display_name, COUNT(*) as cnt
FROM openalex.awards.hrsa_awards
GROUP BY funder.display_name
ORDER BY cnt DESC;


In [ ]:
%sql
-- Check ID uniqueness after row-level keying.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.hrsa_awards;


In [ ]:
%sql
-- Check funder_scheme distribution.
SELECT funder_scheme, COUNT(*) as cnt, SUM(amount) as total_funding
FROM openalex.awards.hrsa_awards
WHERE funder_scheme IS NOT NULL
GROUP BY funder_scheme
ORDER BY cnt DESC
LIMIT 20;


In [ ]:
%sql
-- §6.3 Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    SUM(amount) as total_funding,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(start_date), COUNT(*)) * 100.0, 1) as pct_with_start_date,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description
FROM openalex.awards.hrsa_awards;


In [ ]:
%sql
-- §6.7 amount/currency fail-fast check.
-- HRSA is a grant pattern. pct_amount should be >50%; local source validation found 100% amount coverage.
SELECT
    COUNT(*) AS total,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_amount_nonzero,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(CASE WHEN amount > 0 THEN amount END), 0) AS avg_nonzero_amt
FROM openalex.awards.hrsa_awards;


In [ ]:
%sql
-- Check year distribution.
SELECT start_year, COUNT(*) as cnt, SUM(amount) as total_funding
FROM openalex.awards.hrsa_awards
WHERE start_year IS NOT NULL
GROUP BY start_year
ORDER BY start_year DESC;


In [ ]:
%sql
-- Raw grantee summary for QA only. Grantee is not part of the final awards schema.
SELECT
    grantee_name,
    grantee_state_abbreviation,
    COUNT(*) as cnt,
    SUM(TRY_CAST(REGEXP_REPLACE(financial_assistance, '[^0-9.-]', '') AS DOUBLE)) as total_funding
FROM openalex.awards.hrsa_awards_raw
WHERE grantee_name IS NOT NULL
GROUP BY grantee_name, grantee_state_abbreviation
ORDER BY total_funding DESC
LIMIT 20;
